In [23]:
from pathlib import Path
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

api_key = Path('secrets/entsoe_api_key.txt').read_text(encoding='utf-8').strip()
client = EntsoePandasClient(api_key=api_key)

country_code = 'DE_LU'
tz = 'Europe/Berlin'
now = pd.Timestamp.now(tz=tz).floor('15min')
end = now.normalize() + pd.Timedelta(days=2)  # end of tomorrow

try:
    prices = client.query_day_ahead_prices(
        country_code,
        start=now,
        end=end,
        resolution='15T',
    )
    prices = prices[prices.index >= now]
    prices = prices[~prices.index.duplicated(keep='first')].sort_index()

    prices_df = prices.rename('price_eur_mwh').to_frame()
    prices_df.index.name = 'timestamp'
    prices_df = prices_df.reset_index()

    print(f'{country_code} available future rows: {len(prices_df)}')
    print(prices_df.head(10))
except NoMatchingDataError:
    prices_df = pd.DataFrame(columns=['timestamp', 'price_eur_mwh'])
    print('No future day-ahead prices are published yet.')


DE_LU available future rows: 47
                  timestamp  price_eur_mwh
0 2026-03-10 12:15:00+01:00          88.80
1 2026-03-10 12:30:00+01:00          87.66
2 2026-03-10 12:45:00+01:00          90.99
3 2026-03-10 13:00:00+01:00          93.88
4 2026-03-10 13:15:00+01:00          81.79
5 2026-03-10 13:30:00+01:00          98.63
6 2026-03-10 13:45:00+01:00          89.80
7 2026-03-10 14:00:00+01:00         101.99
8 2026-03-10 14:15:00+01:00          91.26
9 2026-03-10 14:30:00+01:00         103.92


In [24]:
prices_df["price_cent_kwh"] = prices_df['price_eur_mwh'] / 10
prices_df

,timestamp,price_eur_mwh,price_cent_kwh
0,2026-03-10 12:15:00+01:00,88.80,8.880
1,2026-03-10 12:30:00+01:00,87.66,8.766
2,2026-03-10 12:45:00+01:00,90.99,9.099
3,2026-03-10 13:00:00+01:00,93.88,9.388
4,2026-03-10 13:15:00+01:00,81.79,8.179
5,2026-03-10 13:30:00+01:00,98.63,9.863
6,2026-03-10 13:45:00+01:00,89.80,8.980
7,2026-03-10 14:00:00+01:00,101.99,10.199
8,2026-03-10 14:15:00+01:00,91.26,9.126
9,2026-03-10 14:30:00+01:00,103.92,10.392
